# 📘 Notebook 4: Gradient Descent — Derivation, Learning Rate & Convergence

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐ (Intermediate)

---

## 🏠 Why Does This Matter?

You have a house price model. You know which direction to move the weights (the gradient).  
**But how do you actually do the training?**

Gradient descent is the **algorithm** that turns the gradient into weight updates, repeating until the model is good. It is the engine inside:
- Every neural network training loop
- `model.fit()` in Keras
- `optimizer.step()` in PyTorch

Understanding it means you can diagnose training failures, choose learning rates intelligently, and know what "training" actually does.

---

## 📚 Prerequisites
- Notebooks 1–3 (derivatives, chain rule, gradient vectors)

---

## Part 1: The Algorithm

### Plain English First

You are blindfolded on a hilly terrain. You want to reach the valley.

**Your strategy:**
1. Feel the ground beneath your feet — which direction is downhill?
2. Take a step in that direction
3. Stop. Feel the ground again.
4. Repeat until you stop moving (you've reached a flat point — the valley)

That's gradient descent. Mathematically:

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \cdot \nabla L(\mathbf{w}_t)$$

Where:
- `w_t` = current weights
- `α` (alpha) = **learning rate**: how large a step to take
- `∇L(w_t)` = gradient at current weights (which direction is uphill)
- `-` sign = we go **downhill** (opposite to gradient)

### Why Does This Decrease the Loss?

From the Taylor expansion (first-order approximation):

$$L(\mathbf{w} - \alpha\nabla L) \approx L(\mathbf{w}) - \alpha \|\nabla L\|^2$$

Since `α > 0` and `‖∇L‖² ≥ 0`, the new loss is always **smaller** than the old loss  
— as long as `α` is small enough. Each step is **guaranteed** to reduce the loss!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────
# IMPLEMENT gradient descent from scratch
#
# We'll use our house price loss:
#   L(w1, w2) = (w1 - 2)² + (w2 + 1)²
#   ∇L = [2(w1-2), 2(w2+1)]
# Optimal weights: w1=2.0, w2=-1.0
# ─────────────────────────────────────────────────────────────────

def loss(w):
    """House price loss. w is a 2D vector [w1, w2]."""
    return (w[0] - 2)**2 + (w[1] + 1)**2

def gradient(w):
    """Gradient of the loss. Returns a 2D vector."""
    return np.array([2*(w[0] - 2),   # ∂L/∂w1
                     2*(w[1] + 1)])  # ∂L/∂w2

def gradient_descent(loss_fn, grad_fn, w_start, learning_rate, n_steps):
    """
    Run gradient descent for n_steps iterations.

    loss_fn       : function that takes w and returns a scalar loss
    grad_fn       : function that takes w and returns the gradient vector
    w_start       : initial weight vector (starting point)
    learning_rate : step size α (how far to move each step)
    n_steps       : how many steps to take

    Returns: path (all weight values), losses (all loss values)
    """
    w      = np.array(w_start, dtype=float)   # copy to avoid modifying original
    path   = [w.copy()]                        # record all visited weight values
    losses = [loss_fn(w)]                      # record loss at each step

    for step in range(n_steps):
        g = grad_fn(w)                  # step 1: compute gradient at current w
        w = w - learning_rate * g       # step 2: move opposite to gradient
        path.append(w.copy())           # record new position
        losses.append(loss_fn(w))       # record new loss

    return np.array(path), np.array(losses)


# ─── Run with 4 different learning rates and compare ──────────────
w_start       = np.array([-0.5, 1.5])   # starting weights (far from optimum)
n_steps       = 50
learning_rates= [0.001, 0.01, 0.1, 1.0]
colors        = ['royalblue', 'green', 'darkorange', 'crimson']

results = {}    # store results for each learning rate
for lr in learning_rates:
    path, losses = gradient_descent(loss, gradient, w_start, lr, n_steps)
    results[lr] = {'path': path, 'losses': losses}
    final_loss = losses[-1]
    status = 'converged ✓' if final_loss < 0.001 else ('diverged ✗' if final_loss > losses[0] else 'slow...')
    print(f"lr={lr:5}: final loss = {final_loss:.6f}  → {status}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualize: convergence paths on loss contour + loss curves
# ─────────────────────────────────────────────────────────────────

# Build the loss surface backdrop
w1v = np.linspace(-2, 5, 200)
w2v = np.linspace(-3, 4, 200)
W1, W2 = np.meshgrid(w1v, w2v)
Z = (W1 - 2)**2 + (W2 + 1)**2

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: convergence paths on the contour map ──────────────────
axes[0].contourf(W1, W2, Z, levels=20, cmap='viridis', alpha=0.6)
axes[0].contour(W1, W2, Z,  levels=10, colors='white', alpha=0.3, linewidths=0.5)

for lr, color in zip(learning_rates, colors):
    path = results[lr]['path']
    # Only plot points that are within the visible range (diverging paths go off screen)
    visible = np.all((path[:,0] > -2) & (path[:,0] < 5) &
                     (path[:,1] > -3) & (path[:,1] < 4), axis=0)
    # Use a boolean mask per row instead
    mask = ((path[:,0] > -2) & (path[:,0] < 5) &
            (path[:,1] > -3) & (path[:,1] < 4))
    vpath = path[mask]
    if len(vpath) > 1:
        axes[0].plot(vpath[:,0], vpath[:,1], '-o',
                     color=color, markersize=4, linewidth=1.8,
                     label=f'lr={lr}', alpha=0.9)

axes[0].plot(*w_start, 'ws', markersize=14, label='Start', zorder=10)
axes[0].plot(2, -1, 'r*', markersize=20, label='Optimum (w1=2, w2=-1)', zorder=10)
axes[0].set_xlabel('w₁ (sqft weight)', fontsize=12)
axes[0].set_ylabel('w₂ (bedroom weight)', fontsize=12)
axes[0].set_title('Gradient Descent Paths\nDifferent learning rates lead to very different behavior', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].set_xlim(-2, 5); axes[0].set_ylim(-3, 4)

# ── Right: loss over iterations ──────────────────────────────────
for lr, color in zip(learning_rates, colors):
    losses = results[lr]['losses']
    losses_clipped = np.clip(losses, 1e-12, 1e4)   # clip for log scale display
    axes[1].semilogy(losses_clipped, color=color, linewidth=2.5, label=f'lr={lr}')

axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Loss  (log scale)', fontsize=12)
axes[1].set_title('Loss vs Training Steps\n(log scale reveals convergence rate)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Add text annotations explaining each learning rate behavior
axes[1].text(30, 1e-4, 'lr=0.1:\nFast convergence', color='darkorange', fontsize=8)
axes[1].text(30, 1e1,  'lr=1.0:\nDiverges!', color='crimson', fontsize=8)

plt.suptitle('Gradient Descent: Effect of Learning Rate on House Price Model',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Learning rate summary:")
print("  lr=0.001 — converges, but needs ~10,000 steps (too slow for big models!)")
print("  lr=0.01  — converges well in ~100 steps")
print("  lr=0.1   — fast convergence in ~20 steps  ← sweet spot for this problem")
print("  lr=1.0   — step is too large, overshoots, loss EXPLODES")

## Part 2: Batch vs SGD vs Mini-batch

### Plain English First

In a real dataset you have 1 million houses. Computing the gradient over ALL 1 million houses before taking one step would be extremely slow.

Three strategies:

| Strategy | How many houses per gradient? | Pros | Cons |
|----------|------------------------------|------|------|
| **Batch GD** | All N (1 million) | Very accurate gradient | 1 step = 1 full pass over data (slow!) |
| **SGD** | 1 house | Very fast | Noisy gradient; may oscillate |
| **Mini-batch GD** | B houses (e.g., 32–512) | Good balance | Need to choose batch size B |

**Mini-batch gradient descent** is what everyone uses in practice.  
When people say "SGD" in ML, they almost always mean mini-batch SGD.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Compare Batch GD vs SGD vs Mini-batch GD
# Task: learn weights for house price prediction (50 houses, 2 features)
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N = 200    # number of houses

# Generate synthetic house dataset
X_data = np.random.randn(N, 2)              # features: (sqft_norm, beds_norm)
w_true = np.array([2.0, -1.0])             # the weights we want to recover
y_data = X_data @ w_true + 0.3 * np.random.randn(N)   # prices with noise

def full_mse_loss(w):
    """MSE loss using ALL N training examples."""
    return np.mean((X_data @ w - y_data)**2)

def batch_gradient(w):
    """Gradient using ALL N examples. Accurate but slow."""
    return 2 * X_data.T @ (X_data @ w - y_data) / N

def mini_batch_gradient(w, batch_indices):
    """
    Gradient using only the houses in batch_indices.

    batch_indices : array of indices (e.g., [5, 12, 31, ...] for a batch of 32 houses)
    """
    Xb = X_data[batch_indices]    # features for this batch only
    yb = y_data[batch_indices]    # prices for this batch only
    B  = len(batch_indices)       # batch size
    return 2 * Xb.T @ (Xb @ w - yb) / B    # same formula, but over B examples

def run_training(batch_size, lr, n_epochs):
    """
    Train with a given batch size.
    One epoch = one full pass over all N training examples.

    batch_size=N  → Batch GD (one gradient per epoch)
    batch_size=1  → SGD (N gradients per epoch)
    batch_size=32 → Mini-batch GD (N/32 gradients per epoch)
    """
    w          = np.zeros(2)   # start at [0, 0]
    epoch_losses = []          # loss at the end of each epoch
    indices    = np.arange(N)  # indices of all training examples

    for epoch in range(n_epochs):
        np.random.shuffle(indices)    # shuffle data order each epoch
        for start in range(0, N, batch_size):     # iterate over mini-batches
            batch = indices[start : start + batch_size]   # get this batch's indices
            g     = mini_batch_gradient(w, batch)         # compute gradient on batch
            w     = w - lr * g                            # update weights
        epoch_losses.append(full_mse_loss(w))    # record full loss after each epoch
    return w, epoch_losses

n_epochs = 80
w_batch, losses_batch = run_training(N,  lr=0.1,  n_epochs=n_epochs)
w_mini,  losses_mini  = run_training(32, lr=0.05, n_epochs=n_epochs)
w_sgd,   losses_sgd   = run_training(1,  lr=0.01, n_epochs=n_epochs)

# ─── Plot comparison ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_batch, 'royalblue', linewidth=2.5, label=f'Batch GD (B=N={N})')
axes[0].plot(losses_mini,  'green',     linewidth=2.5, label='Mini-batch GD (B=32)')
axes[0].plot(losses_sgd,   'crimson',   linewidth=1.5, label='SGD (B=1)', alpha=0.8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Convergence: Batch vs Mini-batch vs SGD', fontsize=12)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(losses_batch, 'royalblue', linewidth=2.5, label=f'Batch GD')
axes[1].semilogy(losses_mini,  'green',     linewidth=2.5, label='Mini-batch GD')
axes[1].semilogy(losses_sgd,   'crimson',   linewidth=1.5, label='SGD', alpha=0.8)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss (log scale)')
axes[1].set_title('Same (log scale) — notice SGD noise', fontsize=12)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('House Price Training: Batch Size Comparison (200 houses, 2 weights)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"True weights:      w = {w_true}")
print(f"Batch GD learned:  w = {w_batch.round(3)}")
print(f"Mini-batch learned:w = {w_mini.round(3)}")
print(f"SGD learned:       w = {w_sgd.round(3)}")

## Part 3: Convergence — When Does Training Stop?

### Plain English First

How do you know when the model has finished learning?

**Common stopping criteria:**
1. **Gradient is tiny:** `‖∇L‖ < ε` — the surface is flat, we're near a minimum
2. **Loss stops changing:** `|L_new - L_old| < ε` — no more improvement
3. **Fixed number of epochs:** Most practical approach — just run for N steps and stop
4. **Validation loss increases:** We're overfitting — time to stop (early stopping)

**For convex functions** (like MSE loss): gradient descent is **guaranteed** to converge to the global minimum if `α` is small enough.

**Convergence rate:** Each step, the excess loss decreases by a constant fraction → convergence is **exponential** (looks like a straight line on a log scale).

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Convergence analysis: track gradient magnitude and loss gap
# ─────────────────────────────────────────────────────────────────

w      = np.array([-0.5, 1.5], dtype=float)   # starting weights
lr     = 0.1
L_star = 0.0      # true minimum loss (since L = (w1-2)² + (w2+1)², min = 0)

history = {'step': [], 'loss': [], 'grad_norm': [], 'excess_loss': [], 'w1': [], 'w2': []}

for step in range(60):
    g    = gradient(w)                          # compute gradient
    L    = loss(w)                              # current loss

    # Record metrics before the update
    history['step'].append(step)
    history['loss'].append(L)
    history['grad_norm'].append(np.linalg.norm(g))   # how large is the gradient?
    history['excess_loss'].append(L - L_star)         # how far from optimal?
    history['w1'].append(w[0])
    history['w2'].append(w[1])

    w = w - lr * g    # gradient descent update

# ─── Plot four metrics of convergence ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].plot(history['loss'], 'steelblue', linewidth=2)
axes[0,0].set_title('Loss over Training Steps\n(linear scale)', fontsize=11)
axes[0,0].set_xlabel('Step'); axes[0,0].set_ylabel('Loss')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].semilogy(history['excess_loss'], 'steelblue', linewidth=2)
axes[0,1].set_title('Excess Loss L−L* over Steps\n(log scale → straight line = exponential decay)', fontsize=11)
axes[0,1].set_xlabel('Step'); axes[0,1].set_ylabel('L − L*  (log scale)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].semilogy(history['grad_norm'], 'tomato', linewidth=2)
axes[1,0].set_title('Gradient Magnitude ‖∇L‖ → 0\n(converging to flat point)', fontsize=11)
axes[1,0].set_xlabel('Step'); axes[1,0].set_ylabel('‖∇L‖  (log scale)')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(history['w1'], 'royalblue', linewidth=2, label='w1 → 2.0')
axes[1,1].plot(history['w2'], 'green',     linewidth=2, label='w2 → -1.0')
axes[1,1].axhline(2.0,  color='royalblue', linestyle='--', alpha=0.5, label='w1 optimal=2.0')
axes[1,1].axhline(-1.0, color='green',     linestyle='--', alpha=0.5, label='w2 optimal=-1.0')
axes[1,1].set_title('Weight Values over Training\n(converging to optimal)', fontsize=11)
axes[1,1].set_xlabel('Step'); axes[1,1].set_ylabel('Weight value')
axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Convergence Analysis: House Price Model (lr=0.1)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print convergence table
print(f"{'Step':6} | {'Loss':12} | {'‖∇L‖':12} | {'w1':8} | {'w2':8}")
print("-" * 55)
for i in range(0, 60, 10):
    print(f"{history['step'][i]:6} | {history['loss'][i]:12.8f} | "
          f"{history['grad_norm'][i]:12.8f} | {history['w1'][i]:8.5f} | {history['w2'][i]:8.5f}")

---

## ✅ Self-Check Questions

**1.** Write the gradient descent update rule in plain English.  
   (No symbols — describe what happens to each weight each step.)

**2.** Your current loss is 5.0 and `‖∇L‖ = 3.0`. After one step with `lr=0.1`,  
   the loss should decrease by approximately how much? (Use the Taylor approximation.)

**3.** Why does using only 1 sample (SGD) lead to a noisy training curve,  
   while using all N samples (batch GD) gives a smooth curve?

**4.** You're training a house price model and the loss is going **up** (not down).  
   What are the two most likely causes?

**5.** The loss curve on a log scale is a **straight line downward**. What does that  
   tell you about the convergence rate?

<details>
<summary>Click to see answers</summary>

1. "At each step, compute how much the loss changes when you increase each weight by a tiny amount. Then adjust each weight in the direction that would decrease the loss, scaled by the learning rate."

2. Decrease ≈ `α × ‖∇L‖² = 0.1 × 9 = 0.9`. New loss ≈ `5.0 - 0.9 = 4.1`.

3. With 1 sample, the gradient is a noisy estimate of the true gradient (it only sees one house, which might be unusual). With N samples, errors average out and the gradient is much more accurate.

4. (a) **Learning rate is too large** — steps overshoot the minimum. (b) **Bug in gradient computation** — gradient might have wrong sign or scaling.

5. A straight line on a log scale means the loss is decreasing **exponentially** — each step reduces the excess loss by the same fraction. This is called **linear convergence** and is expected for gradient descent on convex functions.

</details>

---

## 📌 Summary

| Concept | Formula / Rule | House price meaning |
|---------|--------------|---------------------|
| **Update rule** | `w ← w − α·∇L` | Adjust each weight by a fraction of its gradient |
| **Learning rate α** | Small=slow, large=diverge | How aggressively to update weights each step |
| **Batch GD** | Gradient over all N houses | Stable but slow for large datasets |
| **Mini-batch GD** | Gradient over B houses | Best trade-off (B=32–512 typical) |
| **Convergence** | `‖∇L‖→0` or loss stops changing | Weights have reached their optimal values |

**➡️ Next Notebook:** Convexity, Saddle Points, Local vs Global Minima — understanding the shape of the loss surface and what gradient descent can (and can't) guarantee.